# Tutorial, part 2: reading the fits

This notebook is part 2 of the *hxex* tutorial; the same steps, with more explanation, are in
the user guide: [Tutorial, part 2](https://github.com/tuttlelm/hxex/blob/main/docs/guide/tutorial-fits.md). It continues from the results of
part 1 (`hxex_b5_tutorial_1_basics.ipynb`), in the same folder: run part 1 first.

**How to use it.** Click a cell and press **Shift + Enter** to run it; the next cell is then
selected. Run the cells in order, from the top. A line starting with `!` runs an `hxex`
command, exactly as you would type it in a terminal. Most cells that change `hxex.toml` run
`hxex set`, which changes one setting and prints what it changed; the cells that add rules of
several lines use the small helpers in `tutorial_helpers.py`, beside this notebook. The guide
says how to make each change in a text editor instead.

**The data** are from the HDX-MS experiments of: Woods CN, Janowska MK, Ulmer LD, Kaur Sidhu J,
Stone NL, James EI, Guttman M, Bush MF, Klevit RE. Activation mechanism of small heat shock
protein HSPB5 revealed by disease-associated mutants. *Proc Natl Acad Sci USA* 122(20),
e2425061122 (2025). https://doi.org/10.1073/pnas.2425061122. Please cite it if you use the data.

Run this cell first: it loads *hxex* and the helpers.

In [ ]:
%matplotlib inline
from IPython.display import Image
from tutorial_helpers import add_before, review, show_table

import hxex

## Step 1: three more peptides

This part runs `hxex report` and `hxex analyse` several times, so the project file says once
that each command replaces what it wrote before (`overwrite = true`). Then three peptides to
look at: WT 68-75 (`data_id` 107), WT 42-54 (56), and R120G 114-137 at charge 3 (166).

In [ ]:
!hxex set hxex.toml output.overwrite true
!hxex set hxex.toml figures.series "[107, 56, 166]"

In [ ]:
!hxex report hxex.toml --figures

## Step 2: a very bright peptide

WT 68-75: every spectrum but one is flagged `bright` (see the titles), and the undeuterated
controls are flagged `saturated`. The labelled spectra have flat tops, which the fit describes
as two populations; the one dim spectrum, 30 min replicate 2, has one population.

In [ ]:
results = hxex.load_results("hxex.toml")
hxex.show_series(results, "WT", "68-75", 2, kinds=["spectra"])

`hxex info` tells you where the run's detector stops counting in proportion: its `noise` line
says that undeuterated controls with tops of 400,000 or more were left out of the noise
measurement.

In [ ]:
!hxex info hxex.toml

What the analysis of part 1 made of this peptide: one population at 1 min, 30 min, and 20 h
(`best_set_K`), and the dim spectrum left out as a replicate outlier.

In [ ]:
show_table("results_analysis_WT/species_uptake.csv",
           ["time", "K", "rank", "mean", "mean_se", "fD", "population", "n_spectra",
            "best_set_K", "best_set_reason"], sample="WT", start=68, end=75, role="")
show_table("results_analysis_WT/left_out_spectra.csv", sample="WT", peptide_range="0068-0075")

## Step 3: record a decision in the review table

The review table is a copy of `spectra.csv` with three columns filled in: `review_status`,
`review_K`, and `note`. Here, 30 min replicate 2 of WT 68-75 is marked `accepted`: a spectrum
you accepted is never left out by the analysis' filters.

In [ ]:
review("WT", "0068-0075", 1800, 2, review_status="accepted",
       note="dim, below the saturation onset")
!hxex set hxex.toml review.table user_review_table.csv

## Step 4: what to leave alone

WT 42-54 at 30 min: a small slow population (6 to 10 %) in each replicate. One spectrum alone
cannot pin it down, but the four together can (`best_set_K` 2). Leave it as it is.

In [ ]:
show_table("results_analysis_WT/species_uptake.csv",
           ["time", "K", "rank", "fD", "population", "population_se", "best_set_K"],
           sample="WT", start=42, end=54, role="")

## Step 5: suspect spectra

The `control_notes` of `series.csv` say when a control is best fitted with more than one
population. In R120G 114-137 at charge 3 the undeuterated controls (`m0_multimodal`) have a
second, smaller envelope about 11 Da heavier, and it is in every labelled spectrum too:
another species, not a state of the protein.

In [ ]:
show_table("results_tables/series.csv",
           ["data_id", "sample", "peptide_range", "charge", "n_multi", "control_notes"],
           sample="R120G", peptide_range="0114-0137")
hxex.show_series(results, "R120G", "114-137", 3, kinds=["spectra"])

A rule in `[review]` leaves the whole series out of the analysis (the fits are kept):

In [ ]:
add_before("# [[review.exclude]]", """
[[review.exclude]]
sample = "R120G"
peptide_range = "114-137"
charge = 3
note = "a second species in every spectrum, the undeuterated controls too"
""")

Check that *hxex* reads both decisions, and write the tables, the figures, and the analysis again:

In [ ]:
!hxex check hxex.toml
!hxex report hxex.toml --figures
!hxex analyse hxex.toml

In [ ]:
show_table("results_analysis_WT/species_uptake.csv",
           ["time", "K", "mean", "mean_se", "fD", "n_spectra"],
           sample="WT", start=68, end=75, role="", K=1)

## Step 6: how many populations: the population test

The population test asks, for every spectrum and for the spectra of each peptide and time
together, whether a further population is needed, against simulated spectra with the same
noise. It is the slow step of this part: about an hour on a laptop, about 15 minutes with a GPU.

In [ ]:
!hxex set hxex.toml run.population_test.simulations 20

In [ ]:
!hxex resume hxex.toml
!hxex analyse hxex.toml

In [ ]:
show_table("results_analysis_WT/species_uptake.csv",
           ["time", "K", "rank", "fD", "population", "set_test_K", "best_set_reason"],
           sample="WT", start=42, end=54, role="")

The population tiers: residue by residue, how strongly the peptides covering it show two populations:

In [ ]:
Image("results_analysis_WT/population_tiers.png")

## Step 7: comparing the two populations

For the peptides with two populations in both samples: the positions of the two, and the change
in the share of the low-D one.

In [ ]:
Image("results_analysis_WT/woods_K2_R120G_1min.png")

In [ ]:
Image("results_analysis_WT/residue_significance_K2_R120G.png")

The guide's [Tutorial, part 2](https://github.com/tuttlelm/hxex/blob/main/docs/guide/tutorial-fits.md) explains how to read these
figures, and [Outputs](https://github.com/tuttlelm/hxex/blob/main/docs/guide/outputs.md) describes every table.